### 1. This block initializes the model parameters for the Viterbi algorithm. We define:
- `states`: the possible hidden states in the HMM (E = exon, 5 = donor splice site, I = intron).
- `start_prob`: the initial probability distribution over states (we start in state E).
- `trans_prob`: the transition probabilities between states.
- `emit_prob`: the emission probabilities of observing each nucleotide (A, C, G, T) from each state.

In [6]:
states = ['E', '5', 'I']
start_prob = {'E': 1.0, '5': 0.0, 'I': 0.0}

trans_prob = {
    'E': {'E': 0.9, '5': 0.1, 'I': 0.0},
    '5': {'I': 1.0, 'E': 0.0, '5': 0.0},
    'I': {'I': 0.9, 'E': 0.1, '5': 0.0},
}

emit_prob = {
    'E': dict.fromkeys(['A','C','G','T'], 0.25),  # Equal chance for any base
    '5': {'A': 0.0, 'C': 0.0, 'G': 1.0, 'T': 0.0}, # Only G is allowed at donor site
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4},
}


### 2. Calculate Log Probability of a Given Path

This block defines a helper function to compute the **log probability of an observed sequence** given a specific sequence of states (path), using the model parameters. This is helpful to verify if a path (like the one in the Nature Primer) gives the expected score.

In [7]:
import math

def get_log_prob_of_a_given_path(path, observed, start_prob = start_prob , trans_prob = trans_prob , emit_prob = emit_prob ):
    if len(path) != len(observed):
        raise ValueError("path and observed sequence must have the same length")

    logp = 0.0
    s0 = path[0]
    logp += math.log(start_prob[s0])
    logp += math.log(emit_prob[s0][observed[0]])

    for i in range(1, len(path)):
        prev_state = path[i-1]
        curr_state = path[i]
        logp += math.log(trans_prob[prev_state][curr_state])
        logp += math.log(emit_prob[curr_state][observed[i]])

    return round(logp, 2)

# Example usage
given_path = "EEEEEEEEEEEEEEE5IIIIIIIIII"
observed_seq  = "CTTCATGTGAAAGCAGACGTAAGTCA"
print("Log probability of given path:", get_log_prob_of_a_given_path(given_path, observed_seq))

Log probability of given path: -40.23


### 3. Viterbi Algorithm Implementation

This code block implements the **Viterbi algorithm**, which finds the most probable sequence of states (path) that could have produced the observed nucleotide sequence.

Key steps:
- Initialization: Set up the first time step with starting probabilities.
- Recursion: For each time step, compute the highest probability path to each state.
- Termination: Choose the path with the highest probability at the final time step.

We use `math.log` to avoid numerical underflow when multiplying many small probabilities. A small pseudocount is also added to handle zero-probability transitions or emissions.

In [8]:
def viterbi(obs, states, start_p, trans_p, emit_p):
    V = [{}]  # Viterbi matrix: list of dicts
    path = {}
    pseudocount = 1e-10  # Small value to avoid log(0)

    # Initialization
    for s in states:
        V[0][s] = math.log(start_p.get(s, 0) + pseudocount) + math.log(emit_p[s].get(obs[0], 0) + pseudocount)
        path[s] = [s]

    # Recursion
    for t in range(1, len(obs)):
        V.append({})
        new_path = {}

        for curr in states:
            max_prob = float('-inf')
            best_prev = None

            for prev in states:
                prob = V[t-1][prev] + math.log(trans_p[prev].get(curr, 0) + pseudocount) + math.log(emit_p[curr].get(obs[t], 0) + pseudocount)
                if prob > max_prob:
                    max_prob = prob
                    best_prev = prev

            V[t][curr] = max_prob
            if best_prev is not None:
                new_path[curr] = path[best_prev] + [curr]
            else:
                # fallback in case no best_prev was found
                new_path[curr] = [curr] * (t + 1)

        path = new_path

    # Termination
    final_state = max(V[-1], key=V[-1].get)
    return path[final_state]


# Example Viterbi run
most_likely_path = viterbi(observed_seq, states, start_prob, trans_prob, emit_prob)
print("Most likely state path:", ''.join(most_likely_path))


Most likely state path: EEEEEEEEEEEEEEEEEEEEEEEEEE
